In [1]:
# ============================================================
# 1. 라이브러리 및 환경 설정
# ============================================================

import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================

train = pd.read_csv("../../data/interim/train_missing_handled.csv")
test = pd.read_csv("../../data/interim/test_missing_handled.csv")

train_fe = train.copy()
test_fe = test.copy()

os.makedirs("../../output/processed", exist_ok=True)

In [30]:
# ============================================================
# 3. Title 파생변수 생성
# ============================================================

train_fe['Title'] = train_fe['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_fe['Title'] = test_fe['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)


print("[train_fe 확인]\n",train_fe['Title'].value_counts())
print("\n [test_fe 확인]\n",test_fe['Title'].value_counts())

# 동의어 통합
title_map = {
    'Mlle' : 'Miss',
    'Ms' : 'Miss',
    'Mme' : 'Mrs'
}
train_fe['Title'] = train_fe['Title'].replace(title_map)
test_fe['Title'] = test_fe['Title'].replace(title_map)

print("\n[map - train_fe]\n",train_fe['Title'].value_counts())
print("\n [map - test_fe]\n",test_fe['Title'].value_counts())

# rare 추출 
title_counts = train_fe['Title'].value_counts()
rare_titles = title_counts[title_counts < 10].index
print("\n[rare_titles 확인]\n", rare_titles)

train_fe['Title'] = train_fe['Title'].replace(rare_titles, 'Rare')
test_fe['Title'] = test_fe['Title'].replace(rare_titles, 'Rare')

valid_titles = set(train_fe['Title'].unique())

test_fe['Title'] = test_fe['Title'].apply(
    lambda x: x if x in valid_titles else 'Rare'
)

print("\n[rare 적용 - train]\n", train_fe['Title'].value_counts())
print("\n[rare 적용 - test]\n",test_fe['Title'].value_counts())

[train_fe 확인]
 Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64

 [test_fe 확인]
 Title
Mr        240
Miss       78
Mrs        72
Master     21
Col         2
Rev         2
Ms          1
Dr          1
Dona        1
Name: count, dtype: int64

[map - train_fe]
 Title
Mr          517
Miss        185
Mrs         126
Master       40
Dr            7
Rev           6
Major         2
Col           2
Don           1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

 [map - test_fe]
 Title
Mr        240
Miss       79
Mrs        72
Master     21
Col         2
Rev         2
Dr          1
Dona        1
Name: count, dtype: int64

[rare_titles 확인]
 Index(['Dr', 'Rev', 'Major', 'Col', 'Don', 'Lady',

In [31]:
# ============================================================
# 4. 가족 관련 파생변수 생성
# ============================================================

train_fe['FamilySize'] = train_fe['SibSp'] + train_fe['Parch'] + 1
test_fe['FamilySize'] = test_fe['SibSp'] + test_fe['SibSp'] + 1

train_fe['IsAlone'] = np.where(train_fe['FamilySize'] == 1, 1, 0)
test_fe['IsAlone'] = np.where(test_fe['FamilySize'] == 1, 1, 0)

In [32]:
# ============================================================
# 5. AgeBand 파생변수 생성
# - 04_bivariate_analysis에서 비교한 후보 중 
#   해석 가능성과 표본 수를 고려한 구간 기준을 적용한다. 
# ============================================================

age_bins = [0, 10, 20, 40, 60, 80]
age_labels = ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']

train_fe['AgeBand'] = pd.cut(
    train_fe['Age'],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

test_fe['AgeBand'] = pd.cut(
    test_fe['Age'],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

In [33]:
# ============================================================
# 6. Fare 관련 파생변수 생성
# - Fare는 오른쪽으로 치우친 분포이므로 log1p 변환을 추가한다.
# - FareBand는 train 기준 qcut 경계를 만든 뒤 test에도 같은 기준을 적용한다.
# ============================================================

train_fe['Fare_log'] = np.log1p(train_fe['Fare'])
test_fe['Fare_log'] = np.log1p(test_fe['Fare'])

fare_bins = pd.qcut(
    train_fe['Fare'],
    q=4,
    retbins=True,
    duplicates='drop'
)[1]

fare_bins[0] = -np.inf
fare_bins[-1] = np.inf

fare_labels_all = ['Low', 'MidLow', 'MidHigh', 'High']
fare_labels = fare_labels_all[:len(fare_bins) -1 ]

train_fe['FareBand'] = pd.cut(
    train_fe['Fare'],
    bins=fare_bins,
    labels=fare_labels,
    include_lowest=True
)

test_fe['FareBand'] = pd.cut(
    test_fe['Fare'],
    bins=fare_bins,
    labels=fare_labels,
    include_lowest=True
)

In [34]:
# ============================================================
# 7. CabinDeck 파생변수 생성
# ============================================================

train_fe['CabinDeck'] = train_fe['Cabin'].fillna('Missing').str[0]
test_fe['CabinDeck'] = test_fe['Cabin'].fillna('Missing').str[0]

In [45]:
# ============================================================
# 8. TicketPrefix 파생변수 생성
# ============================================================

train_fe['TicketPrefix'] = (
    train_fe['Ticket']
        .astype(str)
        .str.replace(r'[0-9./]','',regex=True)
        .str.strip()
)

test_fe['TicketPrefix'] = (
    test_fe['Ticket']
        .astype(str)
        .str.replace(r'[0-9./]','',regex=True)
)
train_fe['TicketPrefix'] = train_fe['TicketPrefix'].replace('','NONE')
test_fe['TicketPrefix'] = test_fe['TicketPrefix'].replace('','NONE')

ticket_prefix_counts = train_fe['TicketPrefix'].value_counts()
rare_ticket_prefixes = ticket_prefix_counts[ticket_prefix_counts < 10].index

train_fe['TicketPrefix'] = train_fe['TicketPrefix'].replace(rare_ticket_prefixes, 'Rare')
test_fe['TicketPrefix'] = test_fe['TicketPrefix'].replace(rare_ticket_prefixes, 'Rare')

valid_ticket_prefixes = set(train_fe['TicketPrefix'].unique())

test_fe['TicketPrefix'] = test_fe['TicketPrefix'].apply(
    lambda x: x if x in valid_ticket_prefixes else 'Rare'
)

In [46]:
# ============================================================
# 9. 생성 결과 저장
# ============================================================

train_fe.to_csv("../../data/processed/train_fe.csv", index=False)
test_fe.to_csv("../../data/processed/test_fe.csv", index=False)

In [54]:
# ============================================================
# 10. 생성 결과 확인
# ============================================================

new_features = [
    'Title',
    'FamilySize',
    'IsAlone',
    'AgeBand',
    'Fare_log',
    'FareBand',
    'CabinDeck',
    'TicketPrefix'
]

print("[train_fe 파생변수 확인]")
display(train_fe[new_features].head())

print("\n[test_fe 파생변수 확인]")
display(test_fe[new_features].head())

print("\n[train_fe 파생변수 결측값 확인]")
display(train_fe[new_features].isnull().sum())

print("\n[test_fe 파생변수 결측값 확인]")
display(test_fe[new_features].isnull().sum())

print("\n[train_fe shape]")
print(train_fe.shape)

print("\n[test_fe shape]")
print(test_fe.shape)

print("\nFeature Engineering 완료")

[train_fe 파생변수 확인]


,Title,FamilySize,IsAlone,AgeBand,Fare_log,FareBand,CabinDeck,TicketPrefix
0,Mr,2,0,YoungAdult,2.110213,Low,M,A
1,Mrs,2,0,YoungAdult,4.280593,High,C,PC
2,Miss,1,1,YoungAdult,2.188856,MidLow,M,STONO
3,Mrs,2,0,YoungAdult,3.990834,High,C,NONE
4,Mr,1,1,YoungAdult,2.202765,MidLow,M,NONE



[test_fe 파생변수 확인]


,Title,FamilySize,IsAlone,AgeBand,Fare_log,FareBand,CabinDeck,TicketPrefix
0,Mr,1,1,YoungAdult,2.178064,Low,M,NONE
1,Mrs,3,0,Adult,2.079442,Low,M,NONE
2,Mr,1,1,Senior,2.369075,MidLow,M,NONE
3,Mr,1,1,YoungAdult,2.268252,MidLow,M,NONE
4,Mrs,3,0,YoungAdult,2.586824,MidLow,M,NONE



[train_fe 파생변수 결측값 확인]


Title           0
FamilySize      0
IsAlone         0
AgeBand         0
Fare_log        0
FareBand        0
CabinDeck       0
TicketPrefix    0
dtype: int64


[test_fe 파생변수 결측값 확인]


Title           0
FamilySize      0
IsAlone         0
AgeBand         0
Fare_log        0
FareBand        0
CabinDeck       0
TicketPrefix    0
dtype: int64


[train_fe shape]
(891, 20)

[test_fe shape]
(418, 20)

Feature Engineering 완료
